# Auditoria de Limpeza de Dados: sales_data
    
Este notebook contém a trilha de auditoria automatizada das operações de limpeza avançada realizadas no dataset **sales_data**.

**Pipeline Aplicado:**
1. Ingestão de Dados Brutos
2. Tratamento de Valores Ausentes (Missing Values)
3. Limitação de Outliers (Winsorization) para estabilidade financeira
4. Normalização de Variáveis (Z-Score)
5. Exportação da Base Lapidada (Ready for Modeling)

Gerado via *Fábrica de Ciência de Dados (AntiGravity)* em 2026-04-09 15:56:22.
    

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

# Configurações visuais (padrão global)
pd.set_option('display.max_columns', None)


## 1. Ingestão de Dados

In [ ]:
raw_path = r"C:\Users\luizn\OneDrive\Área de Trabalho\Projetos Ciencia de dados\data\raw\sales_data.csv"
processed_path = r"C:\Users\luizn\OneDrive\Área de Trabalho\Projetos Ciencia de dados\data\processed\sales_data_limpo.csv"

# Carregamento
df = pd.read_csv(raw_path, encoding='utf-8', sep=None, engine='python')
print(f"Shape original: {df.shape}")
df.head()


## 2. Tratamento de Valores Ausentes
Imputação estratégica: Mediana para variáveis contínuas numéricas e Moda para variáveis categóricas, preservando ao máximo os registros originais para cálculos de ROI.

In [ ]:
missing_before = df.isnull().sum().sum()

# Numéricas: preenchidas com mediana
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Categóricas e Texto: preenchidas com moda
cat_cols = df.select_dtypes(exclude=[np.number]).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

missing_after = df.isnull().sum().sum()
print(f"Valores Ausentes Antes: {missing_before} | Depois: {missing_after}")


## 3. Tracer de Outliers e Limitação (Winsorize)
Aplicando restrições nos limites de 5% e 95% para variáveis com forte variação. Evitamos descarte para que não haja perda de clientes ou de registros transacionais valiosos.

In [ ]:
from scipy.stats.mstats import winsorize

# Aplicaremos apenas em colunas que possuem variabilidade significativa para evitar collapse de range
for col in num_cols:
    if df[col].nunique() > 10:
        # Winsorize nos percentis 5% e 95%
        # Usando clip invés de winsorize puro para controle do dataframe Pandas
        lower = df[col].quantile(0.05)
        upper = df[col].quantile(0.95)
        df[col] = df[col].clip(lower=lower, upper=upper)

print("Ajustes de limites por Winsorization aplicados.")


## 4. Normalização Padrão (Scaling)
Aplicando o StandardScaler em colunas numéricas que não possuam características de identificadores (IDs).

In [ ]:
scaler = StandardScaler()

# Filtrar colunas que provavelmente não são IDs
feat_cols = [c for c in num_cols if 'id' not in c.lower() and df[c].nunique() > 2]

if feat_cols:
    df[feat_cols] = scaler.fit_transform(df[feat_cols])
    print(f"Normalização concluída em {len(feat_cols)} colunas numéricas.")
else:
    print("Nenhuma coluna apta para normalização detectada.")


## 5. Exportação

In [ ]:
# Garantir a pasta raiz do processed
os.makedirs(os.path.dirname(processed_path), exist_ok=True)
df.to_csv(processed_path, index=False)
print(f"Arquivo sanitizado exportado para: {processed_path} | Shape final: {df.shape}")
